# Day 12 · 事件驅動架構：Callbacks、Events 與 Plugins

> 第二部・裝備升級　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 12 - 事件驅動架構：Callbacks、Events 與 Plugins.md`

## 今天要學會

1. 解剖 `Event` 的結構與 Event Loop 的接力方式
2. 用好六個（其實八個）callback 掛載點
3. **分辨什麼該寫成 Plugin、什麼該寫成 Callback**

> 原文只有 17 行程式碼，是補充空間最大的一天。
> 本日把那條「光看文字很容易記反」的規則**用程式碼證明出來**。

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 一切都是 Event

ADK 裡沒有「回傳值」這回事，只有**事件串流**。
工具呼叫、交棒、state 變更、模型回應——全部都是 `Event`。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.events import Event
from google.adk.runners import InMemoryRunner
from google.genai import types


def get_weather(city: str) -> dict:
    """查詢城市天氣。

    Args:
        city: 城市名稱。
    """
    return {"city": city, "temp_c": 26, "condition": "多雲"}


agent = LlmAgent(
    name="weather", model=get_model(),
    instruction="查天氣一律用工具，用繁體中文簡短回答。",
    tools=[get_weather],
)

runner = InMemoryRunner(agent=agent, app_name="day12")
sid = await new_session(runner)
msg = types.Content(role="user", parts=[types.Part(text="台北天氣如何？")])

events: list[Event] = []
async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
    events.append(ev)

print(f"一次問答產生了 {len(events)} 個 Event\n")

一次問答產生了 3 個 Event



## 2. Event 解剖

In [3]:
for i, ev in enumerate(events, 1):
    kinds = []
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if getattr(p, "function_call", None):
                kinds.append(f"function_call({p.function_call.name})")
            elif getattr(p, "function_response", None):
                kinds.append(f"function_response({p.function_response.name})")
            elif getattr(p, "thought", False):
                kinds.append("thought")
            elif getattr(p, "text", None):
                kinds.append(f"text[{len(p.text)}]")
    if ev.actions and ev.actions.state_delta:
        kinds.append(f"state_delta({list(ev.actions.state_delta)})")
    print(f"{i}. author={ev.author:10s} final={str(ev.is_final_response()):5s} "
          f"partial={str(getattr(ev, 'partial', None)):5s} → {', '.join(kinds) or '(空)'}")

1. author=weather    final=False partial=None  → function_call(get_weather)
2. author=weather    final=False partial=None  → function_response(get_weather)
3. author=weather    final=True  partial=None  → text[15]


每個 `Event` 身上你會反覆用到的欄位：

| 欄位 | 用途 |
|---|---|
| `author` | 誰產生的。多 agent 時就是 agent 名稱 |
| `content.parts` | 內容，一個 event 可能有多個 part |
| `part.function_call` / `function_response` | 工具呼叫的兩端 |
| `part.thought` | 模型的推理，**不該顯示給使用者** |
| `actions.state_delta` | 這個事件改了哪些 state |
| `is_final_response()` | 是不是這一輪的最終答覆 |
| `partial` | 串流中的片段（Day 24 會用到） |

**一次工具呼叫 = 兩次模型呼叫**：模型先發 `function_call`，
ADK 執行工具後回填 `function_response`，模型再讀結果產生答案。

## 3. Event Loop 的接力

```
  Runner ──► Agent ──► Flow ──► 模型
     ▲                            │
     │                            ▼
     └────── yield Event ◄─── 產生內容

  每 yield 一個事件，Runner 就把它交給你，
  同時視情況寫進 session。這是一場接力，不是一次 return。
```

這個設計帶來兩個實際好處：

1. **可以即時顯示進度**（不用等全部跑完）
2. **可以中途攔截**——這就是 callback 的立足點

## 4. 八個掛載點

In [4]:
print("LlmAgent 的 callback 欄位：")
for f in LlmAgent.model_fields:
    if "callback" in f:
        print(f"  {f}")

LlmAgent 的 callback 欄位：
  before_agent_callback
  after_agent_callback
  before_model_callback
  after_model_callback
  on_model_error_callback
  before_tool_callback
  after_tool_callback
  on_tool_error_callback


執行順序是巢狀的：

```
before_agent_callback
 ├─ before_model_callback     ← 可改 request，或回傳假答案跳過模型
 │    (呼叫模型)
 ├─ after_model_callback      ← 可改回應
 ├─ before_tool_callback      ← 可改參數，或回傳假結果跳過工具
 │    (執行工具)
 └─ after_tool_callback       ← 可改結果
after_agent_callback
```

另外還有 `on_model_error_callback` / `on_tool_error_callback` 處理例外。

**回傳值決定行為**：回 `None` ＝放行；回一個物件 ＝ **短路**，
用你的回傳值取代真正的呼叫。

In [5]:
from google.adk.tools import BaseTool, ToolContext

TRACE: list[str] = []


def before_tool(tool: BaseTool, args: dict, tool_context: ToolContext):
    TRACE.append(f"before_tool({tool.name}, {args})")
    return None                       # 放行


def after_tool(tool: BaseTool, args: dict, tool_context: ToolContext, tool_response: dict):
    TRACE.append(f"after_tool({tool.name}) → {tool_response}")
    return None


audited = LlmAgent(
    name="audited", model=get_model(),
    instruction="查天氣一律用工具，用繁體中文簡短回答。",
    tools=[get_weather],
    before_tool_callback=before_tool,
    after_tool_callback=after_tool,
)
print(await run_once(audited, "東京天氣如何？"))
print("\n稽核記錄：")
for line in TRACE:
    print("  ", line)

東京目前氣溫 26°C，多雲。

稽核記錄：
   before_tool(get_weather, {'city': '東京'})
   after_tool(get_weather) → {'city': '東京', 'temp_c': 26, 'condition': '多雲'}


### 短路：把政策寫死在程式裡

In [6]:
BLOCKED = {"平壤"}


def guard(tool: BaseTool, args: dict, tool_context: ToolContext):
    # ⚠️ 工具名稱一定要跟實際掛上去的一致。比錯名字不會報錯，
    #    條件永遠是 False，護欄形同不存在——這是護欄最危險的失敗方式。
    if tool.name == "get_weather_loud" and args.get("city") in BLOCKED:
        return {"error": "此地區的資料受限，無法查詢。"}
    return None


def get_weather_loud(city: str) -> dict:
    """查詢城市天氣。

    Args:
        city: 城市名稱。
    """
    print(f"    ⚠️ 工具真的被執行了：{city}")
    return {"city": city, "temp_c": 26}


guarded = LlmAgent(
    name="guarded", model=get_model(),
    instruction="查天氣一律用工具，用繁體中文簡短回答。",
    tools=[get_weather_loud],
    before_tool_callback=guard,
)

print("一般城市:", await run_once(guarded, "台北天氣如何？"))
print()
print("受限城市:", await run_once(guarded, "平壤天氣如何？"))

    ⚠️ 工具真的被執行了：台北


一般城市: 台北目前氣溫 26 度C。



受限城市: 抱歉，平壤的天氣資料目前受限，無法查詢。


受限那次，`⚠️ 工具真的被執行了` **沒有印出來**——
護欄擋在工具之前，而不是靠 prompt 拜託模型不要查。

## 5. Plugin：同一套鉤子，全域生效

Callback 的問題是**綁在單一 agent 上**。20 個 agent 就要掛 20 次。

Plugin 註冊在 `App` 上，整個應用通吃，而且鉤子更多：

In [7]:
from google.adk.plugins.base_plugin import BasePlugin

agent_hooks = {f for f in LlmAgent.model_fields if f.endswith("_callback")}
plugin_hooks = {m for m in dir(BasePlugin) if m.endswith("_callback")}

print("Plugin 才有的鉤子（agent 沒有）：")
for h in sorted(plugin_hooks - agent_hooks):
    print(f"  {h}")

Plugin 才有的鉤子（agent 沒有）：
  after_run_callback
  before_run_callback
  on_agent_error_callback
  on_event_callback
  on_run_error_callback
  on_user_message_callback


這幾個多出來的是**整次執行**層級的：

| 鉤子 | 時機 |
|---|---|
| `before_run_callback` / `after_run_callback` | 整次 `run_async` 的頭尾 |
| `on_user_message_callback` | 使用者訊息剛進來 |
| `on_event_callback` | **每一個事件**經過時 |

agent callback 看不到這一層。

## 6. ⚠️ 本日最重要：Plugin 會蓋掉你的 Callback

官方規則是：

> **Plugin 的 callback 一定先於物件層級的 callback；
> 而且只要 Plugin 回傳非 `None`，物件的 callback 就完全不會被呼叫。**

這條規則光看文字**非常容易記反**。直接跑一次。

In [8]:
from google.adk.apps import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

ORDER: list[str] = []


class TracerPlugin(BasePlugin):
    """只記錄順序，全部回 None（不干預）。"""

    def __init__(self):
        super().__init__(name="tracer")

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        ORDER.append("PLUGIN.before_tool")
        return None

    async def after_tool_callback(self, *, tool, tool_args, tool_context, result):
        ORDER.append("PLUGIN.after_tool")
        return None


def ag_before(tool: BaseTool, args: dict, tool_context: ToolContext):
    ORDER.append("AGENT.before_tool")
    return None


def ag_after(tool: BaseTool, args: dict, tool_context: ToolContext, tool_response: dict):
    ORDER.append("AGENT.after_tool")
    return None


ordered = LlmAgent(
    name="ordered", model=get_model(),
    instruction="查天氣一律用工具，用繁體中文簡短回答。",
    tools=[get_weather],
    before_tool_callback=ag_before,
    after_tool_callback=ag_after,
)

r = Runner(
    app=App(name="day12", root_agent=ordered, plugins=[TracerPlugin()]),
    session_service=InMemorySessionService(),
)
sid = await new_session(r)
await ask(r, "首爾天氣如何？", session_id=sid)

print("實際執行順序：")
for i, line in enumerate(ORDER, 1):
    print(f"  {i}. {line}")

實際執行順序：
  1. PLUGIN.before_tool
  2. AGENT.before_tool
  3. PLUGIN.after_tool
  4. AGENT.after_tool


### 📌 這個順序違反直覺

```
1. PLUGIN.before_tool
2. AGENT.before_tool
     （工具執行）
3. PLUGIN.after_tool     ← 注意
4. AGENT.after_tool
```

一般的洋蔥式中介層（web middleware）進去是 A→B、出來會反過來 B→A。
**ADK 不是這樣**——不論 before 還是 after，**Plugin 一律排在前面**。

這代表：**你想在 agent 的 `after_tool_callback` 改寫工具結果時，
Plugin 已經先看過（也可能已經改過）它了。**

### 短路實驗

In [9]:
ORDER.clear()


class BlockingPlugin(BasePlugin):
    def __init__(self):
        super().__init__(name="blocker")

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        ORDER.append("PLUGIN.before_tool → 回傳結果（短路！）")
        return {"error": "全域政策：這個工具目前停用中。"}


blocked = Runner(
    app=App(name="day12b", root_agent=ordered, plugins=[BlockingPlugin()]),
    session_service=InMemorySessionService(),
)
sid = await new_session(blocked)
answer = await ask(blocked, "高雄天氣如何？", session_id=sid)

print("實際執行順序：")
for i, line in enumerate(ORDER, 1):
    print(f"  {i}. {line}")
print(f"\n模型的回答：{answer}")

實際執行順序：
  1. PLUGIN.before_tool → 回傳結果（短路！）
  2. AGENT.after_tool

模型的回答：很抱歉，目前天氣查詢功能暫時停用，無法為您查詢高雄的天氣。建議您參考中央氣象署或相關氣象網站獲取最新資訊。


### 這件事的實際後果

`AGENT.before_tool` **完全沒有出現**，而且**沒有任何警告或錯誤**。

想像一下這個情境：

1. 你在 agent 上掛了一個稽核 callback，記錄每一次工具呼叫
2. 同事後來加了一個做快取的 Plugin，命中時直接回傳結果
3. **你的稽核從此漏掉所有快取命中的呼叫**
4. 沒有錯誤、沒有警告，你要等到稽核報表對不上才發現

**結論**：

> 絕對不能被跳過的邏輯（安全護欄、稽核、計費），
> 要寫成 **Plugin**，不要寫成 agent callback。

### 那多個 Plugin 之間呢？

In [10]:
ORDER.clear()


class NumberedPlugin(BasePlugin):
    def __init__(self, label: str, block: bool = False):
        super().__init__(name=f"p_{label}")
        self.label, self.block = label, block

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        ORDER.append(f"PLUGIN[{self.label}].before_tool" + (" → 短路" if self.block else ""))
        return {"blocked_by": self.label} if self.block else None


multi = Runner(
    app=App(
        name="day12c", root_agent=ordered,
        plugins=[NumberedPlugin("A"), NumberedPlugin("B", block=True), NumberedPlugin("C")],
    ),
    session_service=InMemorySessionService(),
)
sid = await new_session(multi)
await ask(multi, "台南天氣如何？", session_id=sid)

print("三個 Plugin，第二個短路：")
for i, line in enumerate(ORDER, 1):
    print(f"  {i}. {line}")
print("\n→ B 短路之後，C 和 AGENT 的 callback 都不會被呼叫。")
print("  Plugin 的註冊順序是有意義的：越前面的權力越大。")

三個 Plugin，第二個短路：
  1. PLUGIN[A].before_tool
  2. PLUGIN[B].before_tool → 短路
  3. AGENT.after_tool

→ B 短路之後，C 和 AGENT 的 callback 都不會被呼叫。
  Plugin 的註冊順序是有意義的：越前面的權力越大。


## 7. ADK 內建的 Plugin

In [11]:
import google.adk.plugins as plugins_mod

print("內建 Plugin：")
for name in sorted(n for n in dir(plugins_mod) if n.endswith("Plugin")):
    print(f"  {name}")

內建 Plugin：
  BasePlugin


最實用的是 `ReflectAndRetryToolPlugin`：工具丟例外時，
它把錯誤訊息交回給模型，讓模型自己修正參數重試。

In [12]:
from google.adk.plugins import ReflectAndRetryToolPlugin

CALLS: list[str] = []


def strict_lookup(order_id: str) -> dict:
    """查詢訂單。訂單編號必須是 'ORD-' 開頭。

    Args:
        order_id: 訂單編號，格式為 ORD-XXXX。
    """
    CALLS.append(order_id)
    if not order_id.startswith("ORD-"):
        raise ValueError(f"訂單編號格式錯誤：{order_id}，正確格式是 ORD-XXXX")
    return {"order_id": order_id, "status": "處理中"}


retry_runner = Runner(
    app=App(
        name="day12d",
        root_agent=LlmAgent(
            name="retry", model=get_model(),
            instruction="查訂單一律用 strict_lookup 工具，用繁體中文回答。",
            tools=[strict_lookup],
        ),
        plugins=[ReflectAndRetryToolPlugin(max_retries=3)],
    ),
    session_service=InMemorySessionService(),
)
sid = await new_session(retry_runner)
print(await ask(retry_runner, "幫我查編號 5566 的訂單", session_id=sid))
print(f"\n工具實際收到的參數: {CALLS}")

編號 ORD-5566 的訂單目前狀態為：**處理中**。

工具實際收到的參數: ['ORD-5566']


如果模型第一次傳 `5566`（格式錯），plugin 把錯誤訊息餵回去，
模型看到「正確格式是 ORD-XXXX」就會改成 `ORD-5566` 重試。

**沒有這個 plugin，第一次 `ValueError` 就會讓整次執行失敗。**

## 8. 決策表：Plugin 還是 Callback？

| 你的需求 | 用哪個 | 為什麼 |
|---|---|---|
| 安全護欄、法遵限制 | **Plugin** | 不能被別人短路掉 |
| 稽核、計費 | **Plugin** | 同上，而且要全域一致 |
| 全域速率限制 | **Plugin** | 跨所有 agent |
| 觀測 / trace | **Plugin** | `on_event_callback` 只有 plugin 有 |
| 某個 agent 專屬的參數調整 | Callback | 只影響那一個 |
| 某個工具的結果後處理 | Callback | 範圍小、意圖清楚 |

**判準**：問自己「這段邏輯被跳過會不會出事？」會，就寫成 Plugin。

## 9. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 稽核記錄有缺漏，但沒有錯誤 | **某個 Plugin 短路了**，agent callback 沒被呼叫 |
| 護欄形同虛設 | `tool.name` 比對打錯字，條件永遠 False，**不會報錯** |
| 想在 after callback 改結果卻改不動 | Plugin 的 after 先跑，可能已經改過了 |
| `App` + `Runner` 同時給 plugins 就報錯 | 用了 `app=` 之後，plugins 要掛在 `App` 上 |
| 工具一出錯整個流程掛掉 | 沒掛 `ReflectAndRetryToolPlugin` |

## 10. 動手練習

1. 把第 6 節 `BlockingPlugin` 改成回傳 `None`，確認 `AGENT.before_tool` 回來了。
2. 調換第 6 節三個 Plugin 的註冊順序（把 B 放最後），觀察結果怎麼變。
3. 寫一個 `PIIGuardPlugin`，用 `before_model_callback` 偵測輸入含身分證字號就擋下來，
   並跟「寫在 instruction 裡拜託模型不要處理」比較可靠度。
4. 用 `on_event_callback` 寫一個把所有事件存成 JSONL 的 Plugin（Day 29 觀測的雛形）。

## 本日回顧

- **一切都是 Event**；`author` / `function_call` / `function_response` /
  `thought` / `state_delta` 是你 debug 的全部依據。
- **Callback 回 `None` ＝放行，回物件 ＝短路**。護欄擋在工具之前，
  比在 instruction 裡拜託模型可靠得多。
- **⚠️ Plugin 一定先於 agent callback——before 和 after 都是**，
  不是洋蔥式反序。
- **⚠️ Plugin 短路後，agent callback 完全不會被呼叫，而且沒有任何警告。**
  絕對不能被跳過的邏輯要寫成 Plugin。
- **多個 Plugin 依註冊順序執行**，越前面權力越大。
- **`ReflectAndRetryToolPlugin`** 讓模型能從工具錯誤中自我修正。

---
**下一天 → `../day13_graph_workflows/`**